# Does anomaly detection with path signatures add information?

**The question.** Aayush's detector (distance of a window's signature from a "calm" average) flags windows that are followed by high volatility, but so do GARCH and a plain volatility z-score. That test rewards volatility clustering. Here we ask whether signature-based anomaly scores tell us something *beyond* what a volatility model already knows.

**How a detector is built.**
- Each row scores the latest window (1 hour: 1 day of 5-minute bars; 1 day: 7 days of hourly bars).
- The window becomes a path: cumulative return, volume share, range and signed flow. Optionally, the path is divided by the volatility measured *before* the window, so the detector sees shape, not level.
- The path is summarised four ways: **signature** (depth 2–4), a **lag bank** with the same number of columns, hand-built **stats**, and **raw** block sums.
- **Mahalanobis** (expected-signature distance), **Isolation Forest** and an **autoencoder** learn "normal" from the training rows and score each new window.

**How a detector is judged.**
- A *missed event* is one of the 20% of rows where realized volatility most exceeded the HAR-RV forecast.
- The main number is the AUC of the score for predicting missed events, computed within calm, medium and volatile buckets (*stratified*). 0.5 means no information beyond HAR.
- Configs are chosen on the first folds and confirmed on the last one, with purged walk-forward folds throughout.

**The gate.** Signatures count as useful only if the best signature setup beats the best signature-free setup, including the vol baselines, on the selection folds **and** on the last fold (block bootstrap, p < 0.05). If not, we stop and report the negative result.

In [ ]:
import sys, importlib, logging
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, str(Path.cwd().parent))  # repo root, so `anomaly` and Mihika/ import
import anomaly
from anomaly import config as C, features, detectors, evaluate, run as RUN, report as R
for _m in (C, features, detectors, evaluate, RUN, R):
    importlib.reload(_m)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s", datefmt="%H:%M:%S")
pd.set_option("display.float_format", "{:.3f}".format)

## 1. Configuration

In [ ]:
QUICK = False  # True: small grid on 2023+ data to check that everything runs

cfg = C.AnomalyConfig(
    horizons=("1h", "1d"),
    start="2017-08-01",                  # Binance 1-minute klines, cached under data/klines
    target_rows={"1h": 20000, "1d": None},
    output_dir="results/anomaly",
    data_dir="data",
)
if QUICK:
    cfg = cfg.quick()
print(len(cfg.rep_configs()), "path recipes per horizon")
cfg

## 2. Run

Each horizon runs in its own process. Finished path recipes are saved, so an interrupted run resumes where it stopped (delete `results/anomaly/<horizon>/` to redo one).

In [ ]:
root = RUN.run_all(cfg, subprocess_per_horizon=True)
res = R.load_results(root)
pd.DataFrame(res.meta).T

## 3. The gate: do signatures beat everything signature-free?

`Sig/Free select` are stratified surprise AUCs on the selection folds, and `confirm` the same on the last fold. `Last-fold diff` is signature minus signature-free, with a block-bootstrap 95% CI and a one-sided p-value.

In [ ]:
display(R.gate_table(res).style.format(precision=3))

## 4. Leaderboard

Top configs per horizon by stratified surprise AUC on the selection folds. `AUC_level` and `Fwd_vol_ratio` are Aayush-style metrics (they reward volatility clustering), shown for comparison.

In [ ]:
display(R.leaderboard(res, top=12).style.format(precision=3))

## 5. Which representation and detector work?

In [ ]:
R.plot_kind_heatmap(res); plt.show()

## 6. Synthetic check: shape-only anomalies

For test windows, the same returns are re-ordered into a V or an inverted V. Volatility, net move, skew and kurtosis are unchanged; only the path's shape differs. This shows what each selected detector *can* see in principle. It is not evidence about real markets.

In [ ]:
display(R.synthetic_table(res).style.format(precision=3))

## 7. Scores over time

Score percentiles of the selected detectors, with the labeled crises shaded (a case study only).

In [ ]:
for hk in res.meta:
    R.plot_timeline(res, hk); plt.show()

## 8. What next

- **Gate passed on a horizon:** Phase 2 extends to 5m and 7d, adds the HMM, and uses the selected signature detector in the volatility forecasts as a feature, a residual correction and a regime gate, compared against the best signature-free detector.
- **Gate failed:** the honest result is that signature anomaly scores carry no information beyond volatility clustering at these horizons. Only the single residual-correction test is run in the volatility forecasts.